<a href="https://colab.research.google.com/github/josephineabioye/msc-skincare-reaction-prediction/blob/main/notebooks/03_reaction_label_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## Notebook Setup Cell

from google.colab import userdata, drive
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = 'josephineabioye'
REPO_NAME = 'msc-skincare-reaction-prediction'

if not os.path.exists(REPO_NAME):
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
    %cd {REPO_NAME}
else:
    %cd {REPO_NAME}
    !git pull

!git config user.email "josephineabioye@yahoo.com"
!git config user.name "Josephine Abioye"

drive.mount('/content/drive', force_remount=False)

DRIVE_PROJECT = '/content/drive/MyDrive/msc-skincare-project'
DATA_RAW = f'{DRIVE_PROJECT}/data/raw'
DATA_PROCESSED = f'{DRIVE_PROJECT}/data/processed'

print(f"Working in: {os.getcwd()}")

/content/msc-skincare-reaction-prediction
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/msc-skincare-reaction-prediction


In [2]:
## Load consolidated skincare reviews

import pandas as pd
skincare_reviews = pd.read_parquet(f"{DATA_PROCESSED}/skincare_reviews.parquet")

print("Loaded skincare reviews:", len(skincare_reviews))
print("Columns:", skincare_reviews.columns.tolist())

Loaded skincare reviews: 1094411
Columns: ['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd']


In [3]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [10]:
import spacy
from spacy.matcher import PhraseMatcher
from sklearn.metrics import cohen_kappa_score

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
nlp.add_pipe("sentencizer")

# droping the columns not required
skincare_reviews = skincare_reviews.drop(columns=["Unnamed: 0", "author_id"], errors="ignore")

# merging review_title + review_text columns
skincare_reviews["review_full"] = ((skincare_reviews["review_title"].fillna("") + ". " + skincare_reviews["review_text"].fillna(""))
    .str.replace("’", "'", regex=False).str.replace("‘", "'", regex=False).str.strip()
)

# Lexicons: From the lexicon library I created
#Strong Indicators
STRONG = [
    "irritation","irritated","irritating","irritate","burning","burns","burned","burn","stinging","stung","sting","stings","itching","itchy","itchiness","rash",
    "hives","blister","blistering","allergic reaction","allergy","swollen","swelling","welts","chemical burn","contact dermatitis","raw skin","peeled my skin",
    "skin peeled off","skin barrier damaged","damaged my barrier","compromised barrier","barrier damage","skin barrier ruined","damaged my skin","wrecked my barrier",
    "compromised my skin barrier","stripped my skin","stripped all the moisture","over-exfoliated","over exfoliated","sensitized my skin","skin became sensitive",
    "broke me out","caused a breakout","made me break out","gave me acne","gave me pimples","clogs my pores","clogged my pores","clogging my pores","more acne",
    "worse acne","acne got worse","more pimples","hyperpigmentation got worse","dark spots got worse","texture got worse","painful pimples","deep pimples",
    "eyes burned","eyes sting","watery eyes","swollen eyelids","itchy eyes","lip irritation","lip swelling","broke out","broke out my"
]

#Dual use terms
DUAL_USE = [
    "acne","pimples","zits","breakout","break outs","breaking out","redness","red patches","red spots","dry","dryness","dehydrated","oily","oiliness","greasy","shiny",
    "sensitive","sensitivity","bumpy","bumps","clogged pores","pore-clogging","pore clogging","congested skin","congestion","congested","flaky","flaking","scaling",
    "peeling","texture","rough","rough texture","purging","tight","tightness","comedones","whiteheads","blackheads","closed comedones","papules","pustules",
    "nodules","cysts","cystic acne","fungal acne","comedogenic","discoloration","hyperpigmentation","dark spots","puffiness","inflamed","inflammation","flare-up",
    "flare up","eczema flare","chapped lips","cracked lips","break out"
]

#Causal / Worsening phrases that trigger dual use terms
CAUSAL = [
    "caused","made me","made my","left me with","resulted in","after using","since using","every time i use","whenever i use","within hours","immediately after",
    "the next day","triggered","led to","gave me","ended up with","because of","due to","from using",
]
WORSENING = ["worse","worsen","worsened","got worse","aggravate","aggravated","increased"]

# Negation cues / words checked in a window before a match
NEG_CUES = {"no","not","never","without","didn't","doesn't","wasn't","isn't","don't","none","cannot","can't","nor","n't","non"}

# Purpose cues / words checked in a window before a match
PURPOSE_CUES = {"for","help","helps","helping","treat","treats","treating","target","targets","soothe","soothes","soothing","calm","calms","calming","reduce",
                "reduces","reducing","prevent","prevents","combat","combats","fight","fights","clear","fade","fades"}

WINDOW = 5

def _matcher(terms):
    m = PhraseMatcher(nlp.vocab, attr="LOWER")
    m.add("X", [nlp.make_doc(t) for t in terms])
    return m

strong_m, dual_m, causal_m, worsen_m = _matcher(STRONG), _matcher(DUAL_USE), _matcher(CAUSAL), _matcher(WORSENING)

def _suppressed(sd, start):
    window = [t.lower_ for t in sd[max(0, start - WINDOW):start]]
    negated = any(w in NEG_CUES or w.startswith("non-") for w in window)
    purpose = any(w in PURPOSE_CUES for w in window)
    return negated or purpose


SKINTYPE_HEADS = {"oily","dry","sensitive","combination","combo","normal"}
def label_review(doc):
    for sent in doc.sents:
        sd = sent.as_doc()
        for _, start, _ in strong_m(sd):
            if not _suppressed(sd, start):
                return 1
        dual = dual_m(sd)
        if dual and (causal_m(sd) or worsen_m(sd)):
            for _, start, end in dual:
                if end < len(sd) and sd[end].lower_ == "skin" and sd[start].lower_ in SKINTYPE_HEADS:
                    continue  # "oily skin", "dry skin" etc. are skin types, not reactions
                if not _suppressed(sd, start):
                    return 1
    return 0

# develop on a sample first
sample = skincare_reviews.sample(5000, random_state=42).copy()
sample["reaction_label"] = [label_review(doc) for doc in nlp.pipe(sample["review_full"], batch_size=200)]
print(sample["reaction_label"].value_counts(normalize=True))

reaction_label
0    0.908
1    0.092
Name: proportion, dtype: float64


In [11]:
import textwrap
print("labelled REACTION (1)")
for t in sample[sample.reaction_label == 1]["review_full"].sample(15, random_state=1):
    print("•", textwrap.shorten(t, 200))
print("\nlabelled NO reaction (0)")
for t in sample[sample.reaction_label == 0]["review_full"].sample(15, random_state=1):
    print("•", textwrap.shorten(t, 200))

labelled REACTION (1)
• Great product!. So far, so good. I normally have puffiness when I wake up, but after using this product I no longer have puffiness. Overall I give it a 10. It doesn't irritate my skin and [...]
• The. Winner.. I have tried them ALL— and found them to be heavily fragranced, or oily, or surprisingly drying, or irritating to my sensitive skin, or over priced. But not this refreshing, pretty [...]
• Best color corrector I've tried to date. I was hesitant to try this due to the price but also I have sensitive, oily skin. However, I'm glad I purchased it. I use this daily after applying my [...]
• . I didn't like the texture or the way my face would feel a bit too oily. But it worked wonders. I used this cream over night. It would reduce the swelling to some of my acne (not take it away) [...]
• clogs pores. after using this cleanser i always break out! Usually i never get any pimples or whiteheads but without fail, each time I use this cleanser my whole face breaks o

In [12]:
def first_trigger(doc):
    for sent in doc.sents:
        sd = sent.as_doc()
        for _, s, e in strong_m(sd):
            if not _suppressed(sd, s):
                return ("STRONG", sd[s:e].text, sent.text)
        dual = dual_m(sd)
        if dual and (causal_m(sd) or worsen_m(sd)):
            for _, s, e in dual:
                if not _suppressed(sd, s):
                    return ("DUAL", sd[s:e].text, sent.text)
    return None

for t in sample[sample.reaction_label == 1]["review_full"].sample(25, random_state=2):
    trig = first_trigger(nlp(t))
    if trig:
        print(f"[{trig[0]}] «{trig[1]}»  ⟵  {trig[2][:130]}")

[STRONG] «deep pimples»  ⟵  While this product did not make all of my whiteheads and deep pimples vanish, it did significantly improve my adult acne with no s
[DUAL] «sensitive»  ⟵  I don't love the consistency of the product and because of my extremely sensitive skin it wasn't working for my daily routine.
[STRONG] «irritating»  ⟵  I recently bought the Tatcha water cream and noticed it really wasn't making a difference and the fragrance was irritating my skin
[STRONG] «burning»  ⟵  I will definitely say that I wouldn't use this if you have sensitive skin, as there was a slight burning when I first put it on.
[STRONG] «Made me break out»  ⟵  Made me break out.
[DUAL] «dry»  ⟵   Unfortunately I found that my skin was left still fairly dry after using for a few days.
[DUAL] «blackheads»  ⟵   I know sometimes things get worse before they get better, so I continued to use it but it never really got rid of my blackheads.
[STRONG] «irritated»  ⟵  It made my skin feel so hydrated and look le

VALIDATION ITERATION LOG
| Round | Change made and reason | Positive rate (5k sample) | Kappa |
|---|---|---|---|
| 1 | Initial tiered rules: strong terms fire on their own with negation and purpose/indication suppression; dual-use terms gated by a causal or worsening cue in the same sentence; title merged with body | 9.72% | Not yet measured |
| 2 | Noticed that curly apostrophes were preventing some negation cues from matching correctly, so apostrophes were normalised. Observed that terms such as puffiness, inflamed/inflammation, flare-up, eczema flare, and chapped/cracked lips could be indication-related rather than direct adverse reactions, so they were moved to the dual-use category. Identified that some reaction phrases were being missed due to variations in wording, so base verb forms (burn, sting, irritate) and active clog-related phrases were added to improve recall. |9.18% | Not yet measured|
| 3 | Noticed that some negation patterns and purpose-related descriptions were still being incorrectly identified as reactions, so "non" was added to the negation cues and -ing purpose forms (soothing, calming, reducing) were included. Observed that skin-type descriptions such as "oily skin", "dry skin", and "sensitive skin" were triggering false positives, so a skin-type suppressor was added. Identified that some breakout-related reactions were being missed due to variations in wording, so "break out" and "broke out" were added to improve recall. Remaining false positives caused by attribution to other causes, weak phrase connections, and distant indication terms were left for the transformer fallback and manual validation. | 9.20% | Not yet measured|